In [1]:
import random

subjects_list = ["calculus", "algebra", "physics", "chemistry"]
styles = ["slow and conceptual", "fast problem-solving", "interactive", "exam-focused"]
reviews_list = [
    "great for weak students",
    "excellent for exam preparation",
    "very patient and clear",
    "best for advanced learners",
    "budget friendly and supportive"
]

tutors = []

for i in range(50):
    tutor = {
        "id": i,
        "name": f"Tutor {chr(65+i)}",
        "subjects": random.sample(subjects_list, 2),
        "teaching_style": random.choice(styles),
        "experience": random.randint(1, 10),
        "rating": round(random.uniform(4.0, 5.0), 1),
        "price": random.randint(100, 800),
        "reviews": random.choice(reviews_list)
    }
    tutors.append(tutor)

len(tutors)

50

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def tutor_to_text(tutor):
    return f"{tutor['subjects']} {tutor['teaching_style']} {tutor['reviews']}"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
for tutor in tutors:
    tutor["embedding"] = model.encode(tutor_to_text(tutor))

len(tutors[0]["embedding"])

384

In [4]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query):
    q_emb = model.encode(query)

    results = []
    for tutor in tutors:
        sim = cosine_similarity(q_emb, tutor["embedding"])
        results.append((tutor, sim))

    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [5]:
results = search("I am weak in calculus and need basics")

for tutor, score in results:
    print(tutor["name"], round(score, 3))

Tutor \ 0.519
Tutor H 0.507
Tutor h 0.496
Tutor E 0.478
Tutor A 0.477
Tutor U 0.468
Tutor a 0.462
Tutor b 0.46
Tutor Y 0.455
Tutor f 0.451
Tutor N 0.439
Tutor _ 0.434
Tutor m 0.429
Tutor g 0.419
Tutor [ 0.417
Tutor o 0.417
Tutor d 0.408
Tutor ^ 0.407
Tutor e 0.406
Tutor L 0.405
Tutor l 0.405
Tutor T 0.393
Tutor V 0.385
Tutor i 0.385
Tutor C 0.384
Tutor Q 0.368
Tutor P 0.362
Tutor I 0.361
Tutor j 0.353
Tutor n 0.352
Tutor k 0.345
Tutor O 0.343
Tutor M 0.339
Tutor r 0.339
Tutor F 0.334
Tutor ] 0.334
Tutor D 0.325
Tutor p 0.318
Tutor G 0.316
Tutor W 0.303
Tutor c 0.299
Tutor K 0.294
Tutor S 0.292
Tutor J 0.291
Tutor ` 0.278
Tutor Z 0.274
Tutor B 0.244
Tutor X 0.243
Tutor q 0.208
Tutor R 0.142


In [6]:
def rank(results):
    ranked = []

    for tutor, sim in results:
        score = (
            0.5 * sim +
            0.2 * (tutor["rating"] / 5) +
            0.2 * (tutor["experience"] / 10) +
            0.1 * (1 - tutor["price"] / 1000)
        )
        ranked.append((tutor, score))

    ranked.sort(key=lambda x: x[1], reverse=True)
    return ranked[:3]

In [7]:
ranked = rank(results)

for tutor, score in ranked:
    print(tutor["name"], round(score, 3))

Tutor E 0.699
Tutor \ 0.689
Tutor H 0.659


In [8]:
def explain(query, ranked):
    responses = []
    
    for tutor, score in ranked:
        explanation = f"""
        {tutor['name']} is recommended because:
        - Teaching style: {tutor['teaching_style']}
        - Experience: {tutor['experience']} years
        - Strength: {tutor['reviews']}
        """
        responses.append(explanation)
    
    return responses

In [9]:
for exp in explain("need slow calculus teacher", ranked):
    print(exp)


        Tutor E is recommended because:
        - Teaching style: interactive
        - Experience: 10 years
        - Strength: great for weak students
        

        Tutor \ is recommended because:
        - Teaching style: slow and conceptual
        - Experience: 8 years
        - Strength: very patient and clear
        

        Tutor H is recommended because:
        - Teaching style: interactive
        - Experience: 8 years
        - Strength: very patient and clear
        


In [10]:
import requests

BASE_URL = "https://ominous-space-chainsaw-pjgvq75jv7wqc6gqv-8080.app.github.dev/"

def upload_tutors():
    for tutor in tutors:
        payload = {
            "id": tutor["id"],
            "vector": tutor["embedding"].tolist(),
            "metadata": {
                "name": tutor["name"],
                "subjects": tutor["subjects"],
                "style": tutor["teaching_style"],
                "experience": tutor["experience"],
                "rating": tutor["rating"],
                "price": tutor["price"],
                "reviews": tutor["reviews"]
            }
        }

        requests.post(f"{BASE_URL}/vectors", json=payload)

upload_tutors()